# Stage 10: Risk Contribution Analytics

This notebook explains not just what weights HRP and HERC assign, but which assets actually drive portfolio risk.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == "10_risk_contribution":
    project_root = project_root.parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analytics import compare_risk_contributions, risk_contribution_table
from src.covariance import CovarianceFactory
from src.dashboard.plots import (
    plot_hrp_herc_risk_comparison,
    plot_risk_contribution_bar,
    plot_weight_vs_risk_contribution,
)
from src.optimization import HERCAllocator, HRPAllocator


## 1. Load Data

In [ ]:
rng = np.random.default_rng(2040)
dates = pd.date_range(start="2021-01-01", periods=504, freq="B")
common_factor = rng.normal(0.0002, 0.005, size=(len(dates), 1))
idiosyncratic = rng.normal(
    loc=0.0004,
    scale=np.array([0.009, 0.011, 0.015, 0.008, 0.013]),
    size=(len(dates), 5),
)
returns_df = pd.DataFrame(
    common_factor + idiosyncratic,
    index=dates,
    columns=["Equity", "IT", "Gold", "Bonds", "Energy"],
)
returns_df.head()


## 2. Compute Covariance Using CovarianceFactory

In [ ]:
covariance_matrix = CovarianceFactory.compute(
    returns_df,
    method="ledoit_wolf",
)
covariance_matrix.round(6)


## 3. Generate HRP Weights

In [ ]:
hrp_weights = HRPAllocator(covariance_method="ledoit_wolf").optimize(returns_df)
hrp_weights


## 4. Generate HERC Weights

In [ ]:
herc_weights = HERCAllocator(covariance_method="ledoit_wolf").optimize(returns_df)
herc_weights


## 5. Compute Risk Contribution Tables

In [ ]:
hrp_risk_table = risk_contribution_table(hrp_weights, covariance_matrix)
herc_risk_table = risk_contribution_table(herc_weights, covariance_matrix)

display(Markdown("### HRP Risk Contribution Table"))
display(hrp_risk_table.round(4))
display(Markdown("### HERC Risk Contribution Table"))
display(herc_risk_table.round(4))


## 6. Compare Weights vs Risk Contribution

In [ ]:
plot_risk_contribution_bar(hrp_risk_table, title="HRP Percentage Risk Contribution").show()
plot_weight_vs_risk_contribution(hrp_risk_table, title="HRP Weight vs Risk Contribution").show()
plot_risk_contribution_bar(herc_risk_table, title="HERC Percentage Risk Contribution").show()
plot_weight_vs_risk_contribution(herc_risk_table, title="HERC Weight vs Risk Contribution").show()


## 7. Compare HRP vs HERC Risk Contribution

In [ ]:
comparison_df = compare_risk_contributions(hrp_weights, herc_weights, covariance_matrix)
comparison_df.round(4)


In [ ]:
plot_hrp_herc_risk_comparison(comparison_df).show()


## 8. Interpret Drawdown Behavior

In [ ]:
highest_hrp_risk = hrp_risk_table.iloc[0]
highest_herc_risk = herc_risk_table.iloc[0]
comparison_gap = comparison_df.iloc[comparison_df['Risk Contribution Difference'].abs().argmax()]

discussion = [
    "### Research Questions",
    f"- HRP's largest risk contributor is `{highest_hrp_risk['Asset']}`.",
    f"- HERC's largest risk contributor is `{highest_herc_risk['Asset']}`.",
    f"- The biggest HRP vs HERC risk-contribution gap appears in `{comparison_gap['Asset']}`.",
    "- Assets with modest capital weights can still dominate portfolio risk when covariance and volatility are high.",
    "- If HERC reduces concentration in percentage risk contribution, that gives a direct explanation for lower drawdowns even if CAGR is slightly lower.",
]

display(Markdown("\n".join(discussion)))
